# 30.04 Вычислительное ядро двуслойной модели

> **Статус:** каноническая исполняемая проверка прямой модели `30.01`.
> Ядро не использует экспериментальные или медицинские данные и не создаёт
> научного вывода о тканях.

Единственная реализация формулы и аналитических производных находится в
`two_layer_model.py`. Этот notebook проверяет эталонные значения, производные,
масштабные инварианты и взаимность. Локальная/глобальная чувствительность и
выбор размеров принадлежат сериям `31` и `32`.

Полный санированный смешанный предшественник сохранён в
`archive/legacy/31.90_Объединённый_анализ_чувствительности.ipynb`.

## 1. Граница модели

- вещественный знаковый передаточный импеданс, а не автоматически модуль;
- точечные коллинеарные электроды на плоской двухслойной среде;
- `L=2a`, `beta=b/a`, обязательно `0 < b < a`;
- эффективные однородные `rho1`, `rho2` и постоянная толщина `h`;
- SI: метры, Ом·м и Ом;
- число членов ряда выбирается удвоением до сходимости импеданса и всех первых
  производных, а не фиксируется универсально.

Gain/offset, фаза, контакт, площадь электродов и реальная анатомия здесь не
моделируются. Их нельзя поглощать изменением `rho` без отдельного допущения.

In [ ]:
from pathlib import Path
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
notebook_root = next((p for p in candidates if (p / "two_layer_model.py").exists()), None)
if notebook_root is None:
    raise FileNotFoundError("two_layer_model.py not found; run from the repository or Colab Notebooks directory")
sys.path.insert(0, str(notebook_root))

from two_layer_model import (
    apparent_resistivity,
    evaluate,
    geometry_from_size,
    transfer_impedance,
    transfer_impedance_coordinates,
)

## 2. Эталонные численные значения

Это самопроверка формулы на синтетической рабочей точке, а не оценка параметров
добровольца. Значения должны совпасть с таблицей `30.01`.

In [ ]:
rho1, h = 5.0, 0.020
expected = {
    (0.050, 15.0): 5.58577,
    (0.050, 25.0): 5.81610,
    (0.090, 15.0): 6.78717,
    (0.090, 25.0): 7.54596,
    (0.140, 15.0): 8.26561,
    (0.140, 25.0): 9.79495,
}

rows = []
for (size, rho2), target in expected.items():
    a, b = geometry_from_size(size)
    z = transfer_impedance(rho1, rho2, h, a, b)
    rho_a = apparent_resistivity(z, a, b)
    assert np.isclose(rho_a, target, rtol=0.0, atol=5e-5)
    rows.append((size * 1000, rho2, z, rho_a))

print("L, мм | rho2, Ом·м | Z, Ом | rho_a, Ом·м")
for row in rows:
    print("%5.0f | %11.1f | %5.2f | %11.5f" % row)

## 3. Производные и масштабные инварианты

Аналитические производные сравниваются с центральными разностями. Два
тождественных следствия однородности:

$$
rac{ho_1 Z_{ho_1}+ho_2 Z_{ho_2}}{Z}=1,
\qquad
rac{aZ_a+bZ_b+hZ_h}{Z}=-1.
$$

In [ ]:
rho1, rho2, h = 5.0, 20.0, 0.020
a, b = geometry_from_size(0.140)
result = evaluate(rho1, rho2, h, a, b)
parameters = [rho1, rho2, h, a, b]
analytic = [result.d_rho1, result.d_rho2, result.d_h, result.d_a, result.d_b]
names = ["rho1", "rho2", "h", "a", "b"]

for index, (name, derivative) in enumerate(zip(names, analytic)):
    step = 1e-6 * parameters[index]
    plus, minus = parameters.copy(), parameters.copy()
    plus[index] += step
    minus[index] -= step
    numeric = (transfer_impedance(*plus) - transfer_impedance(*minus)) / (2 * step)
    assert np.isclose(derivative, numeric, rtol=2e-7, atol=1e-8)
    print(name, derivative, numeric)

resistivity_identity = (rho1 * result.d_rho1 + rho2 * result.d_rho2) / result.z
geometry_identity = (a * result.d_a + b * result.d_b + h * result.d_h) / result.z
assert np.isclose(resistivity_identity, 1.0, atol=1e-10)
assert np.isclose(geometry_identity, -1.0, atol=1e-10)
print("terms:", result.n_terms)

## 4. Взаимность

Роли токовой и потенциальной пар меняются при фиксированных координатах через
общую координатную формулу. Литеральная подстановка `b > a` в замкнутую
формулу не является проверкой взаимности и должна завершаться ошибкой.

In [ ]:
direct = transfer_impedance_coordinates(a, -a, b, -b, rho1, rho2, h)
reciprocal = transfer_impedance_coordinates(b, -b, a, -a, rho1, rho2, h)
closed = transfer_impedance(rho1, rho2, h, a, b)
assert np.isclose(direct, reciprocal, rtol=0.0, atol=1e-10)
assert np.isclose(direct, closed, rtol=0.0, atol=1e-10)

try:
    transfer_impedance(rho1, rho2, h, b, a)
except ValueError:
    pass
else:
    raise AssertionError("invalid geometry b >= a must be rejected")

## 5. Что проверено и что не проверено

**Проверено:** алгебра эталонной формулы, первые производные, сходимость ряда,
однородный предел, масштабные инварианты и взаимность. Те же проверки доступны
как `python -m unittest discover -s tests -p test_two_layer_model.py -v` из
папки `Colab Notebooks`.

**Не проверено этим notebook:** абсолютный масштаб реального канала, тракт
`BASE/RHEO`, фаза, контакт, применимость плоскослоистости, перенос между
последовательными записями и тканевая интерпретация. Эти ворота принадлежат
`10–20`, `31–34`.